# 01. Quickstart: End-to-End Gemini-to-Gemma Distillation Pipeline

This notebook demonstrates how to configure and run the complete 5-stage `distillfw` pipeline:
1. **`dataset_generator`**: Query Gemini 3.5 teacher models with task prompts.
2. **`dataset_formatter`**: Format responses into Gemma chat/reasoning templates and splits.
3. **`model_trainer`**: Distill into a single-node Gemma student model.
4. **`model_evaluator`**: Compare Teacher vs. Base Student vs. Distilled Student.
5. **`model_deployer`**: Deploy the distilled Gemma model to Vertex AI Endpoints.


In [ ]:
from pathlib import Path
from distillfw import (
    DistillationConfig,
    DistillationPipeline,
    GCPConfig,
    TeacherConfig,
    StudentConfig,
    FormatConfig,
    TrainingConfig,
    PromptFormat,
    TrainingAlgorithm,
)

config = DistillationConfig(
    task_id="customer-support-triage-v1",
    description="Distill Gemini 3.5 Flash into Gemma 3 4B for ticket triage",
    gcp=GCPConfig(
        project_id="my-gcp-project",
        location="us-central1",
        bucket_name="my-distillfw-bucket",
    ),
    teacher=TeacherConfig(
        model_id="gemini-3.5-flash",
        temperature=0.2,
        response_logprobs=True,
        logprobs_top_k=20,
    ),
    student=StudentConfig(
        model_id="google/gemma-3-4b-it",
        peft_method="lora",
        lora_r=16,
    ),
    formatting=FormatConfig(
        prompt_format=PromptFormat.CHAT,
    ),
    training=TrainingConfig(
        algorithm=TrainingAlgorithm.SKEW_KL,
        num_epochs=3,
        per_device_batch_size=4,
    ),
)

print("Canonical Task GCS URI:", config.task_uri)
print(config.to_yaml())


In [ ]:
import json, tempfile
from distillfw import LocalConfig

with tempfile.TemporaryDirectory() as tmp:
    local_config = DistillationConfig(
        task_id=config.task_id,
        local=LocalConfig(storage_root=tmp),
        teacher=config.teacher,
        student=config.student,
        formatting=config.formatting,
        training=config.training,
        evaluation=config.evaluation,
        deployment=config.deployment,
    )
    prompts_file = Path(tmp) / "prompts.jsonl"
    prompts_file.write_text(
        "\n".join(
            json.dumps({"prompt": f"Classify ticket #{i}: My invoice was charged twice."})
            for i in range(20)
        )
    )

    pipeline = DistillationPipeline.init_task(
        config=local_config,
        prompts_path=prompts_file,
        teacher_callable=lambda p, _: {"completion": "Category: BILLING | Priority: HIGH"},
        custom_train_fn=lambda train_p, ckpt_d, exp_d: {"train_loss": 0.31, "global_step": 30},
        student_predict_fn=lambda ps: (["Category: BILLING | Priority: HIGH"] * len(ps), [18.5] * len(ps)),
        judge_fn=lambda p, r, s: {"score": 5, "reason": "Exact match with teacher"},
        deploy_fn=lambda tid, uri, cfg: {"endpoint_resource_name": f"projects/my-gcp-project/locations/us-central1/endpoints/{tid}"},
    )
    final_state = pipeline.resume()
    print("Pipeline Final Status:", final_state.status)
    for stage_name, rec in final_state.stages.items():
        print(f"  {stage_name.value:20s} -> {rec.status.value}")
